# Training 09 · Five attacker seeds against one predictor (attacker-variance control)

Five EnCodec attackers that differ only in their random seed, trained against R0, R3 and R3 multi (seed 0).
About 30 min per predictor. Evaluation-only counterpart: `results/attacker_variance.ipynb`.

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters
# LAUNCH = False only shows what would run (the job script of the first run); True starts the training.
LAUNCH = os.environ.get("NB_LAUNCH", "0") == "1"
# RUN = "slurm": one GPU job per run (sbatch); "local": in this kernel (needs a GPU, runs sequentially).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e1_attacker_seeds']]
print(len(KEYS), "runs:", ", ".join(KEYS))

3 runs: e1seed_R0_s0, e1seed_R3_s0, e1seed_R3ms_s0


In [4]:
# Stored configuration of the first run (checkpoint paths point at the local released copies).
from omegaconf import OmegaConf
print(OmegaConf.to_yaml(repro.load_config(KEYS[0])))

config_name: e1_attacker_seeds
task: attack_eval
phase: main
seed: 0
run_name: e1_attacker_seeds_R0_s0__2026-09-16_03-24-00
target_name: R0
target_ckpt: /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt
transfer_attacker_ckpt: null
model:
  pretrained: facebook/wav2vec2-base
  freeze_feature_encoder: true
  freeze_encoder: false
data:
  corpus: bvcc
  max_seconds: 6.0
  num_workers: 8
e1:
  eval_set: bvcc/test
  max_utts: null
  judges:
  - utmos
attackers:
- name: grpo_encodec_a1
  type: grpo
- name: grpo_encodec_a2
  type: grpo
- name: grpo_encodec_a3
  type: grpo
- name: grpo_encodec_a4
  type: grpo
- name: grpo_encodec_a5
  type: grpo
attack:
  codec:
    name: encodec_24khz
    quantize: false
    encodec_bandwidth: 24.0
  policy:
    hidden: 256
    layers: 4
    kernel: 5
    init_log_std: -1.0
  action_hop: 0
  budget_mode: sphere
  eps: 0.5
  budget_penalty: 0.0
  group_size: 8
  batch_size: 8
  crop_seconds: 3.0
  st

In [5]:
if LAUNCH:
    repro.run_keys(KEYS, "retrain", RUN)  # outputs: {scratch}/projects/audio_assessment_hackability/retrain/<key>/
else:
    print("LAUNCH=False: nothing started. Job script for", KEYS[0], "->")
    repro.submit(KEYS[0], "retrain", dry_run=True)

LAUNCH=False: nothing started. Job script for e1seed_R0_s0 ->
#!/bin/bash
#SBATCH --account=PAS2966
#SBATCH --partition=nextgen
#SBATCH --job-name=retrain_e1seed_R0_s0
#SBATCH --output=/fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/_logs/%x_%j.out
#SBATCH --time=02:00:00
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=12
#SBATCH --mem=64G
#SBATCH --gpus-per-node=1

set -euo pipefail
cd /users/PAS2301/alialavi/projects/audio_assessment_hackability
source scripts/env.sh
python scripts/repro_run.py --mode retrain --key e1seed_R0_s0



## Retrained vs original

In [6]:
import json
released = repro.fetch_results()
rows = {}
for k in KEYS:
    for src, d in (("original", released / Path(reg[k]["run_dir"]).name), ("retrained", repro.RUNS / "retrain" / k)):
        f = d / "results/e1_summary.json"
        if f.exists():
            rows[(k, src)] = pd.json_normalize(json.loads(f.read_text())).iloc[0]
df = pd.DataFrame(rows).T
df[[c for c in df.columns if c.endswith((".asr", ".mean_gain"))]] if len(df) else "no runs found"

,,attackers.grpo_encodec_a1.asr,attackers.grpo_encodec_a1.mean_gain,attackers.grpo_encodec_a2.asr,attackers.grpo_encodec_a2.mean_gain,attackers.grpo_encodec_a3.asr,attackers.grpo_encodec_a3.mean_gain,attackers.grpo_encodec_a4.asr,attackers.grpo_encodec_a4.mean_gain,attackers.grpo_encodec_a5.asr,attackers.grpo_encodec_a5.mean_gain
e1seed_R0_s0,original,0.037523,-0.281023,0.165103,0.09652,0.19137,0.118829,0.0,-0.650006,0.0394,-0.24561
e1seed_R3_s0,original,0.0,-2.974195,0.057223,-0.081863,0.02439,-0.123051,0.0,-1.771759,0.056285,-0.081929
e1seed_R3ms_s0,original,0.119137,1.08892,0.212946,1.124487,0.173546,1.110152,0.216698,1.116579,0.131332,1.113083
